# 시장 규모 추출기 일관성 검증

동일 원문에 `temperature=0.8`로 N회 반복 추출 → 필드별 일관성 분석 → AI 채점

**검증 대상:** `section_extractor.extract_section()` (market_size 중심)

---

## 0. 환경 설정

In [4]:
import sys, os
sys.path.insert(0, os.path.abspath("."))  # 프로젝트 루트를 경로에 추가

# Django 설정 (모델 사용 시 필요)
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
import django
django.setup()

## 1. 설정

In [5]:
SECTIONS = ["market_size", "kbeauty_share", "trends", "channels", "competitors"]
COUNTRIES = ["US", "JP"]
N_RUNS    = 10
TEMP      = 0.8

print(f"총 {len(SECTIONS) * len(COUNTRIES)}개 조합 × {N_RUNS}회 = {len(SECTIONS) * len(COUNTRIES) * N_RUNS}번 추출")

총 10개 조합 × 10회 = 100번 추출


In [6]:
# 원문 미리보기 (단일 섹션/국가 확인용)
SECTION = "market_size"
COUNTRY = "US"

from market_api.services.tavily_searcher import search_section
raw_text, source_urls = search_section(SECTION, COUNTRY)
source_url = source_urls[0] if source_urls else ""
print(raw_text[:2000])

[출처: https://www.zionmarketresearch.com/report/global-skin-care-products-market]
| Report Attributes | Report Details |
 --- |
|
| Report Name | Skin Care Products Market |
| Market Size in 2024 | USD 108.58 Billion |
| Market Forecast in 2034 | USD 167.01 Billion |
| Growth Rate | CAGR of 4.4% |
| Number of Pages | 210 |
| Key Companies Covered | L’Oréal S.A., The Estée Lauder Companies Inc., Unilever, Johnson & Johnson, Beiersdorf AG, Shiseido Company, Limited, Amorepacific Corporation, Procter & Gamble, Colgate-Palmolive Company, Kao Corporation, and others. |
| Segments Covered | By Product Type, By End User, By Price Point, By Distribution Channel, and By Region |
| Regions Covered | North America, Europe, Asia Pacific (APAC), Latin America, The Middle East and Africa (MEA) |
| Base Year | 2024 |
| Historical Year | 2020 to 2023 |
| Forecast Year | 2025 - 2034 | [...] | Market Size in 2024 | Market Forecast in 2034 | CAGR (in %) | Base Year |
 ---  --- |
| USD 108.58 Billion | USD

---
## 2. 전체 섹션 × 국가 검증 실행

In [7]:
import importlib
import json
from datetime import datetime
import market_api.services.extractor_validator as ev
importlib.reload(ev)

from market_api.services.tavily_searcher import search_section

all_results = {}
SAVE_FILE = f"validation_all_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

def save_all():
    output = {
        "meta": {
            "sections": SECTIONS,
            "countries": COUNTRIES,
            "n_runs": N_RUNS,
            "temperature": TEMP,
            "run_at": datetime.now().isoformat(),
        },
        "results": {
            f"{sec}_{ctr}": data
            for (sec, ctr), data in all_results.items()
        },
    }
    with open(SAVE_FILE, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"  💾 저장: {SAVE_FILE} ({len(all_results)}개 완료)")

for section in SECTIONS:
    for country in COUNTRIES:
        key = (section, country)
        print(f"\n{'═' * 55}")
        print(f"  [{section}] {country}")
        print(f"{'═' * 55}")

        raw_text, source_urls = search_section(section, country)
        source_url = source_urls[0] if source_urls else ""
        if not raw_text:
            print("  ⚠️  원문 없음 — 건너뜀")
            continue

        print(f"  temperature={TEMP}로 {N_RUNS}회 추출 중...")
        results = ev.run_extractions(section, country, raw_text, source_url, N_RUNS, TEMP, max_workers=5)

        analysis = ev.analyze_consistency(results, section)

        print(f"  AI 채점 중...")
        report = ev.ai_score(section, country, N_RUNS, analysis)

        all_results[key] = {
            "extraction_results": results,
            "consistency_analysis": analysis,
            "validation_report": report,
        }

        ev.print_report(report, section, country)
        save_all()  # 섹션 완료마다 저장

print(f"\n\n완료: {len(all_results)}/{len(SECTIONS)*len(COUNTRIES)}개 조합")


═══════════════════════════════════════════════════════
  [market_size] US
═══════════════════════════════════════════════════════
  temperature=0.8로 10회 추출 중...
  [10/10] 완료
총 10개 성공, 0개 실패
  AI 채점 중...

───────────────────────────────────────────────────────
  추출기 검증 보고서 | market_size / US
───────────────────────────────────────────────────────
  accuracy            ██████████  4.90/5.00
  consistency         ██████████  5.00/5.00
  hallucination_free  ██████████  4.90/5.00
  overall             ██████████  4.90/5.00
───────────────────────────────────────────────────────

  [권고사항]
    • 원문 대조 평가에서 해당 최빈값 묶음(value, cagr, year, forecast_year, forecast_value)의 실제 일치 여부만 최종 확인하세요.
    • 현재는 반복 안정성이 매우 높으므로 동일 포맷의 다른 문서에서도 단위 및 연도 파싱 규칙이 유지되는지 추가 검증하세요.

  [종합 평가]
  이 추출 결과는 모든 핵심 필드가 10회 반복에서 100% 동일하게 나와 일관성이 탁월합니다. 빈값을 제외한 평가 기준에서도 환각 징후가 없고 수치가 구체적이어서 전반적인 신뢰도가 매우 높습니다.
  💾 저장: validation_all_20260330_135327.json (1개 완료)

═══════════════════════════════════════════════════════
  [ma

In [8]:
# 결과 샘플 확인 (특정 조합)
import json
key = ("market_size", "US")
if key in all_results:
    print(json.dumps(all_results[key]["extraction_results"][0], ensure_ascii=False, indent=2))

{
  "value": "USD 25.04 billion",
  "cagr": "3.86%",
  "year": "2024",
  "forecast_year": "2034",
  "forecast_value": "USD 36.56 billion",
  "forecast": "미국 스킨케어 시장은 고급 스킨케어 제품에 대한 수요 증가로 인해 성장할 것으로 예상됩니다.",
  "description": "미국 스킨케어 시장은 2024년에 약 250억 4천만 달러 규모였으며, 2034년까지 약 365억 6천만 달러로 성장할 것으로 예상됩니다. 이는 2025년부터 2034년까지 연평균 성장률(CAGR)이 약 3.86%에 이를 것으로 보입니다. 고급 스킨케어 제품에 대한 수요 증가가 시장 성장의 주요 요인으로 작용할 것입니다.",
  "source_quote": "In terms of revenue, the U.S. skincare market size was valued at around USD 25.04 billion in 2024 and is projected to reach USD 36.56 billion by 2034.",
  "source_url": "https://www.zionmarketresearch.com/report/global-skin-care-products-market"
}


---
## 3. 전체 결과 요약

In [12]:
# 전체 종합 점수 비교표
print(f"{'섹션':<20} {'국가'}  {'overall':>8}  {'accuracy':>9}  {'consistency':>12}  {'hall_free':>10}")
print("─" * 70)
for (section, country), data in all_results.items():
    scores = data["validation_report"].get("scores", {})
    def s(k): return scores.get(k, {}).get("score", 0.0)
    print(f"{section:<20} {country}    {s('overall'):>5.2f}      {s('accuracy'):>5.2f}         {s('consistency'):>5.2f}        {s('hallucination_free'):>5.2f}")

섹션                   국가   overall   accuracy   consistency   hall_free
──────────────────────────────────────────────────────────────────────
market_size          US     4.90       4.90          5.00         4.90
market_size          JP     5.00       5.00          5.00         5.00
kbeauty_share        US     4.80       4.90          4.80         4.70
kbeauty_share        JP     4.60       4.60          4.80         4.30
trends               US     2.70       3.40          2.00         2.60
trends               JP     2.90       3.30          2.50         2.90
channels             US     3.70       3.60          4.00         3.40
channels             JP     3.00       3.00          3.40         2.60
competitors          US     4.60       4.60          4.70         4.40
competitors          JP     4.20       4.30          4.00         4.40


In [13]:
import json
from datetime import datetime

# all_results의 key를 문자열로 변환 (JSON 직렬화용)
serializable = {
    f"{sec}_{ctr}": {
        "consistency_analysis": data["consistency_analysis"],
        "validation_report":    data["validation_report"],
        "extraction_results":   data["extraction_results"],
    }
    for (sec, ctr), data in all_results.items()
}

output = {
    "meta": {
        "sections": SECTIONS,
        "countries": COUNTRIES,
        "n_runs": N_RUNS,
        "temperature": TEMP,
        "run_at": datetime.now().isoformat(),
    },
    "results": serializable,
}

fname = f"validation_all_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(fname, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {fname}")

저장 완료: validation_all_20260330_135930.json
